# Notebook 01 — Open Library API Collector

## Overview
Collects non-western fantasy books from the Open Library API using 52 targeted genre-subject queries.

**Output:** `Data/Raw/non_western_fantasy/ol_genre_first.json`

## 1. Setup

In [1]:
import requests
import json
import time
from pathlib import Path
import yaml

# ── Find repo root via config.yaml ────────────────────────────────────────────
def find_repo_root():
    current = Path().resolve()
    for parent in [current] + list(current.parents):
        if (parent / "config.yaml").exists():
            return parent
    raise FileNotFoundError("config.yaml not found")

REPO_ROOT = find_repo_root()
with open(REPO_ROOT / "config.yaml") as f:
    config = yaml.safe_load(f)

RAW_DIR   = REPO_ROOT / "Data" / "Raw" / "non_western_fantasy"
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH  = REPO_ROOT / config["output_data_raw_non_western_fantasy"]["ol_results"]
CKPT_PATH = RAW_DIR / "ol_genre_first_checkpoint.json"

PAGE_LIMIT = 20
DELAY      = 0.5

print(f"Repo root: {REPO_ROOT}")
print(f"Output:    {OUT_PATH}")

# ── Guard: skip if data already exists ───────────────────────────────────────
if OUT_PATH.exists():
    with open(OUT_PATH) as f:
        all_books = json.load(f)
    print(f"\n Data already exists - {len(all_books):,} books loaded")
    print("Skipping scraping. Delete ol_genre_first.json to re-scrape.")
    raise SystemExit("Done - data already collected")

Repo root: C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations
Output:    C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\Data\Raw\non_western_fantasy\ol_genre_first.json

 Data already exists - 4,364 books loaded
Skipping scraping. Delete ol_genre_first.json to re-scrape.


SystemExit: Done - data already collected

c:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3755: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 2. Query List

In [2]:
GENRE_SUBJECT_QUERIES = [
    ("fantasy africa", "africa"), ("fantasy african", "africa"),
    ("science fiction africa", "africa"), ("fantasy yoruba", "yoruba"),
    ("fantasy igbo", "igbo"), ("fantasy akan", "akan"),
    ("fantasy zulu", "zulu"), ("fantasy orisha", "orisha"),
    ("fantasy anansi", "anansi"), ("afrofuturism", "afrofuturism"),
    ("africanfuturism", "afrofuturism"), ("fantasy chinese", "chinese"),
    ("fantasy japan", "japanese"), ("fantasy japanese", "japanese"),
    ("fantasy korean", "korean"), ("wuxia", "wuxia"),
    ("xianxia", "xianxia"), ("fantasy yokai", "japanese"),
    ("fantasy kitsune", "japanese"), ("fantasy cultivation", "chinese"),
    ("fantasy india", "south_asian"), ("fantasy indian", "south_asian"),
    ("fantasy hindu mythology", "south_asian"), ("fantasy mahabharata", "south_asian"),
    ("fantasy ramayana", "south_asian"), ("fantasy philippine", "filipino"),
    ("fantasy filipino", "filipino"), ("fantasy aswang", "filipino"),
    ("fantasy vietnamese", "southeast_asian"), ("fantasy thai", "southeast_asian"),
    ("fantasy arabian", "middle_eastern"), ("fantasy djinn", "middle_eastern"),
    ("fantasy persian", "middle_eastern"), ("fantasy islamic", "middle_eastern"),
    ("fantasy ottoman", "middle_eastern"), ("fantasy mughal", "south_asian"),
    ("fantasy aztec", "latin_american"), ("fantasy maya", "latin_american"),
    ("fantasy mayan", "latin_american"), ("fantasy inca", "latin_american"),
    ("fantasy latin american", "latin_american"),
    ("science fiction latin american", "latin_american"),
    ("fantasy curandera", "latin_american"), ("fantasy mesoamerican", "latin_american"),
    ("fantasy native american", "indigenous_americas"),
    ("fantasy indigenous", "indigenous_americas"),
    ("fantasy navajo", "indigenous_americas"), ("fantasy lakota", "indigenous_americas"),
    ("fantasy maori", "oceania"), ("fantasy aboriginal", "oceania"),
    ("fantasy pacific islander", "oceania"), ("fantasy polynesian", "oceania"),
]
print(f"Total queries: {len(GENRE_SUBJECT_QUERIES)}")

Total queries: 52


## 3. API Functions

In [3]:
def fetch_ol_page(query, page=1):
    r = requests.get(
        "https://openlibrary.org/search.json",
        params={
            "q": query,
            "fields": "title,author_name,first_publish_year,cover_i,ratings_average,ratings_count,subject,key",
            "limit": 100,
            "offset": (page - 1) * 100,
        },
        timeout=20
    )
    r.raise_for_status()
    return r.json()

def to_record(doc, tag, query):
    cover_i = doc.get("cover_i")
    return {
        "title":          doc.get("title", "").strip(),
        "author":         (doc.get("author_name") or [""])[0],
        "year_published": doc.get("first_publish_year"),
        "cover_url":      f"https://covers.openlibrary.org/b/id/{cover_i}-M.jpg" if cover_i else "",
        "avg_rating":     doc.get("ratings_average"),
        "num_ratings":    doc.get("ratings_count"),
        "subjects":       doc.get("subject", []),
        "source":         "open_library",
        "source_url":     f"https://openlibrary.org{doc.get('key', '')}",
        "source_tag":     tag,
        "query":          query,
        "description":    "",
    }

test = fetch_ol_page("fantasy yoruba", page=1)
print(f"Test: {len(test.get('docs', []))} results")

Test: 3 results


## 4. Main Scrape Loop

In [4]:
if CKPT_PATH.exists():
    with open(CKPT_PATH) as f:
        all_books = json.load(f)
    done_queries = {b.get("query") for b in all_books if b.get("query")}
    print(f"Resuming - {len(all_books):,} books, {len(done_queries)} queries done")
else:
    all_books    = []
    done_queries = set()
    print("Starting fresh")

seen_keys = {b["source_url"] for b in all_books}

for query, tag in GENRE_SUBJECT_QUERIES:
    if query in done_queries:
        continue
    print(f"\n-- {query} ({tag}) --")
    page_num  = 1
    query_new = 0
    while page_num <= PAGE_LIMIT:
        try:
            data = fetch_ol_page(query, page_num)
            docs = data.get("docs", [])
            if not docs: break
            for doc in docs:
                key = f"https://openlibrary.org{doc.get('key', '')}"
                if key in seen_keys: continue
                seen_keys.add(key)
                rec = to_record(doc, tag, query)
                if rec["title"]:
                    all_books.append(rec)
                    query_new += 1
            print(f"  Page {page_num}: {len(docs)} results | {query_new} new | total: {len(all_books):,}")
            if len(docs) < 100: break
            page_num += 1
            time.sleep(DELAY)
        except Exception as e:
            print(f"  Error: {e}")
            time.sleep(5)
            break
    with open(CKPT_PATH, "w") as f:
        json.dump(all_books, f)

with open(OUT_PATH, "w") as f:
    json.dump(all_books, f, indent=2)
print(f"\nDone - {len(all_books):,} books saved")

Starting fresh

-- fantasy africa (africa) --
  Page 1: 100 results | 100 new | total: 100
  Page 2: 100 results | 200 new | total: 200
  Page 3: 27 results | 227 new | total: 227

-- fantasy african (africa) --
  Page 1: 100 results | 86 new | total: 313
  Page 2: 100 results | 173 new | total: 400
  Page 3: 73 results | 238 new | total: 465

-- science fiction africa (africa) --
  Page 1: 100 results | 81 new | total: 546
  Page 2: 95 results | 170 new | total: 635

-- fantasy yoruba (yoruba) --
  Page 1: 3 results | 1 new | total: 636

-- fantasy igbo (igbo) --
  Page 1: 4 results | 3 new | total: 639

-- fantasy akan (akan) --
  Page 1: 3 results | 3 new | total: 642

-- fantasy zulu (zulu) --
  Page 1: 9 results | 3 new | total: 645

-- fantasy orisha (orisha) --
  Page 1: 4 results | 3 new | total: 648

-- fantasy anansi (anansi) --
  Page 1: 19 results | 16 new | total: 664

-- afrofuturism (afrofuturism) --
  Page 1: 62 results | 59 new | total: 723

-- africanfuturism (afrofut

## 5. Audit

In [5]:
import pandas as pd
from collections import Counter
df = pd.DataFrame(all_books)
print(f"Total: {len(df):,}")
for tag, count in Counter(df["source_tag"]).most_common():
    print(f"  {tag:30s}: {count:,}")

Total: 4,357
  japanese                      : 1,755
  chinese                       : 645
  africa                        : 635
  south_asian                   : 474
  latin_american                : 273
  middle_eastern                : 207
  korean                        : 91
  afrofuturism                  : 60
  wuxia                         : 43
  southeast_asian               : 39
  filipino                      : 30
  oceania                       : 28
  indigenous_americas           : 27
  xianxia                       : 21
  anansi                        : 16
  igbo                          : 3
  akan                          : 3
  zulu                          : 3
  orisha                        : 3
  yoruba                        : 1


## Conclusions

- Genre-first query strategy works well for culturally targeted collection
- Asia and Africa & Diaspora dominate due to larger publishing ecosystems
- Many books missing descriptions — enriched in Notebook 03
- Some noise (western authors writing about non-western settings) — filtered in Notebook 03